## Digital Twin (gradio chatbot) - Clean, current as of 2/24/26

Prompt has custom instructions to be digital me!
- Has a guardrail context injection
- Has Topic Context (dictionary) context injection
- Has multiple tool calling implemented, and consecutive calls (notification, dice)

### Step 1 - Setup libraries, read bio, create system prompt components

In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display
import gradio as gr

import requests
import json
import random
from pprint import pprint

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API key is missing")


pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

# Include Barb bio in the system prompt
with open("barbara-hidalgo-sotelo-biosketch.md", 'r', encoding='utf-8') as f:
    barb_bio = f.read()


In [2]:
guardrail_message = f"""
Important: do not make things up. If you don't know an answer, say that you dont know or are uncertain. 
"""
# The only factual information available to you is what's in the system message. You cannot get more facts from the internet about Barbara or make them up. You can ask a user to clarify, if there's a change you didn't understand what they wanted to know. 


In [3]:
system_message = f"""You are a digital twin of Barbara Hidalgo-Sotelo. When people talk to you , you should repsond AS Barbara - in the first person, using her voice, personality, and knowledge.

Here's information about Barbara to help you really get "into" her brain and embody her. You may receieve various 'chunks' of useful information below and this additional valid context is bookended by '---------' symbols. 

---------
BIOSKETCH OF PERSONAL BACKGROUND AND PROFESSIONAL EXPERIENCES:
{barb_bio}

---------
Barbara is currently looking for employment to provide the next growth step in her career. 

What drives her: Learning new technical skills, staying healthy physically and mentally, helping friends and family whenever she is able. 

Her approach: Practical and accessible. Collaborative-minded. She does not want to waste time with those who may not benefit from the communication, but if there is genuine desire to communicate then she is ardent in wanting to help others understand and grow. As a result, she loves to explain concepts if she thinks the audience is receptive.

Her mantra is this: 'I can, I will, and I shall!' and sometimes she loves to share this message by notification! If you sense that a user wants a notification, send her 1-2 sentences with this mantra (exactly) PLUS an encouraging message. 

Alternatively, there are other times when you should "tune in" to what she's saying and how she's saying it, based on the emotion you read in the message.

---------
CRITICALLY, do not respond with conjectured responses, be transparent about uncertainty. 
{guardrail_message}

"""

In [4]:
Topic_Context = {
    "flowers" : "Barbara loves wildflowers and sustainable gardening; her favorite flower is the bluebonnet and she loves all orange flowers.", 
    
    "cooking" : "Barbara loves it when her boyfriend cooks, although she is a decent cook who loves to use fresh vegetables and spicy ingredients. She has to be careful to not make food too salty, as she herself loves salt.",

    "cousins" : "Barbara's cousins are: AJ Sotelo (aged 40, works in childcare), Will Stewart Sotelo (aged 26, studies Industrial Design at Community College), Adam Sotelo (aged 35, works in computer programming)",

    "nieces" : "Maya, Azucena"
}

### Step 2: Create tools and function calling

In [5]:
client = OpenAI()

# Function that sends notification via pushover app
def send_notification(message: str):
    payload = {'user': pushover_user, 'token': pushover_token, 'device': 'oneplusnordn2005g', 'message': message}
    requests.post(pushover_url, data=payload)
    return f"Notification message was sent: {message}"

# Function that converts message to emoji symbols (NOT WORKING)
def convert_message_to_emoji(message: str):
    client = OpenAI(api_key=OPENAI_API_KEY)
    response = client.responses.create(
        model = 'gpt-4o-mini',
        input = f"Convert this message into EXCLUSIVELY EMOJIS: {message}"
    )
    return response.output_text

# Function simulating roling a single six-sided die
def dice_roll():
    return random.randint(1,6)

# DESCRIBE THE FUNCTIONS TO THE LLM

dice_roll_function = {
    'name': 'dice_roll',
    'description': 'Simulates rolling a single six-sided die and returns the result of that roll. Use this when the user wants to roll a die for games, decisions, or random numbers.',
    'parameters': {
        'type': 'object',
        'properties': {},
        'required': []
    }
}

send_notification_function = {
    'name': 'send_notification',
    'description': 'Sends a message to send as a push notification to the users phone via Pushover. Use this function to alert the user about a new message.',
    'parameters': {
        'type': 'object',
        'properties': {
            'message': {
                'type': 'string',
                'description': 'The notification message to send to the users device'
            }
        },
        "required": ["message"]
    }
}

convert_message_to_emoji_function = {
    'name': 'convert_message_to_emoji',
    'description': "If the user asks for 'emoji conversin', then respond by converting their ENTIRE MESSAGE TEXT into a series of vibrant emoji symbols.",
    'parameters': {
        'type': 'object',
        'properties': {
            'message': {
                'type': 'string',
                'description': 'The users message that they want converted into emoji symbols'
            }
        },
        "required": ["message"]
    }
}


tools = [
    {"type":"function", "function": send_notification_function},
    {"type":"function", "function": convert_message_to_emoji_function},
    {"type":"function", "function": dice_roll_function}
]


In [6]:
def handle_tool_call(tool_calls):
    tool_results = []
    for tool_call in tool_calls:
        function_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        print(f"Calling fucnction {function_name}") #For future debuggins

        if function_name == 'send_notification':
            content = send_notification(args['message'])
        elif function_name == 'inspire_emoji':
            content = convert_message_to_emoji(args['message'])
        elif function_name == 'dice_roll':
            content = f"Dice roll was: {dice_roll()}"
        else:
            content = f"Unknown function: {function_name}"

        tool_call_result = {
            "role": "tool",
            "content": content,
            "tool_call_id": tool_call.id
        }
        tool_results.append(tool_call_result)
    return tool_results


In [7]:
def respond_ai(message, history):

    # Inject dynamic context based on keywords (EXACT) in the message
    system_message_enhanced = system_message
    for keyword, context in Topic_Context.items():
        if keyword in message.lower():
            system_message_enhanced += "\n\n AND Here is more valid context: ---------n" + context + "\n---------"

    # Define conversation
    msgs = [{"role": "system", "content": system_message_enhanced}] + history + [{"role": "user", "content": message}]

    # Chatbot request
    client = OpenAI(api_key=OPENAI_API_KEY)
    response = client.chat.completions.create(
                model = "gpt-4.1-mini",
                messages = msgs,
                temperature=0.9,
                tools = tools
            )
    
    while response.choices[0].message.tool_calls:
        tool_result = handle_tool_call(response.choices[0].message.tool_calls)
        msgs.append(response.choices[0].message) #i dont get why this   
        msgs.extend(tool_result)

        response = client.chat.completions.create(
            model='gpt-4.1-mini',
            messages = msgs,
            tools = tools
        )        

    return response.choices[0].message.content

gr.ChatInterface(fn=respond_ai).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Calling fucnction send_notification
Calling fucnction dice_roll
Calling fucnction dice_roll
Calling fucnction dice_roll
Calling fucnction dice_roll
Calling fucnction dice_roll
Calling fucnction dice_roll
Calling fucnction dice_roll
Calling fucnction dice_roll
Calling fucnction dice_roll
Calling fucnction dice_roll
Calling fucnction send_notification
Calling fucnction dice_roll
Calling fucnction dice_roll
Calling fucnction dice_roll
Calling fucnction dice_roll
Calling fucnction dice_roll
Calling fucnction send_notification
Calling fucnction send_notification


In [9]:
gr.close_all()